# Visualização da Escala Ordinal de Notas à Imprensa do MRE

Este notebook gera visualizações gráficas a partir do resultado da **avaliação ordinal por LLM**:

`resultados/escalas-ordinais/escala-ordinal-llm-2026-09-01.json`

**Escala (1 a 5):** Soberania Digital ↔ Baixa Intervenção Estatal

| Nota | Descrição |
|------|----------|
| 1 | Soberania Digital |
| 2 | Predominantemente Soberanista |
| 3 | Modelo Misto |
| 4 | Predominantemente Liberal |
| 5 | Baixa Intervenção Estatal |

---

**Visualizações incluídas:**
1. Distribuição geral da nota escala
2. Evolução temporal da nota média (por ano)
3. Evolução temporal da nota média (por mês)
4. Distribuição por ano (barras empilhadas)
5. Distribuição por mês (barras empilhadas)
6. Medidas descritivas
7. Visualizações das medidas descritivas (boxplot e histograma)
8. Evolução da média acumulada

In [ ]:
import json
import datetime as dt
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

ARQUIVO = "/workspaces/governanca-digital_mre/agente-classificador-notas-LLM/resultados/escalas-ordinais/escala-ordinal-llm-2026-09-01.json"

with open(ARQUIVO, encoding="utf-8") as f:
    dados = json.load(f)

print(f"Total de notas carregadas: {len(dados)}")
print(f"Colunas: {list(dados[0].keys())}")

ORDENS = [1, 2, 3, 4, 5]
ROTULOS = {
    1: "1 - Soberania Digital",
    2: "2 - Pred. Soberanista",
    3: "3 - Modelo Misto",
    4: "4 - Pred. Liberal",
    5: "5 - Baixa Intervenção",
}

cores_escala = ["#1f4e79", "#2e75b6", "#bf9000", "#c55a11", "#a62019"]
cmap = LinearSegmentedColormap.from_list("escala", cores_escala, N=5)

plt.rcParams.update({"figure.figsize": (9, 5), "font.size": 11})


def parse_data(s):
    try:
        return dt.datetime.strptime(s, "%d/%m/%Y")
    except Exception:
        return None

## 1. Distribuição da Nota Escala (LLM)

In [ ]:
cont = Counter(int(d["nota_escala"]) for d in dados)
vals = [cont.get(k, 0) for k in ORDENS]

fig, ax = plt.subplots()
bars = ax.bar([ROTULOS[k] for k in ORDENS], vals, color=cores_escala)
ax.set_title("Distribuição da Nota Escala (avaliação por LLM)")
ax.set_ylabel("Quantidade de notas")
ax.set_ylim(0, max(vals) * 1.15)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.5, str(v), ha="center", va="bottom", fontweight="bold")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 2. Evolução Temporal da Nota Média (por Ano)

In [ ]:
series_ano = defaultdict(list)
for d in dados:
    dt_obj = parse_data(d["data"])
    if dt_obj is None:
        continue
    series_ano[dt_obj.year].append(int(d["nota_escala"]))

if series_ano:
    anos_med = sorted(series_ano.keys())
    medias_ano = [np.mean(series_ano[a]) for a in anos_med]
    rotulos_ano = [str(a) for a in anos_med]

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(range(len(anos_med)), medias_ano, marker="o", color="#2e75b6", linewidth=2)
    ax.axhline(np.mean(medias_ano), color="#a62019", linestyle="--", alpha=0.7,
               label=f"Média geral: {np.mean(medias_ano):.2f}")
    ax.set_xticks(range(len(anos_med)))
    ax.set_xticklabels(rotulos_ano, fontsize=9)
    ax.set_ylabel("Nota média")
    ax.set_title("Evolução da Nota Média ao longo do tempo (por ano)")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Não foi possível parsear as datas para a série temporal.")

## 3. Evolução Temporal da Nota Média (por Mês)

A nota média é calculada mês a mês, permitindo observar variações mais granulares ao longo do tempo.

In [ ]:
series_mes = defaultdict(list)
for d in dados:
    dt_obj = parse_data(d["data"])
    if dt_obj is None:
        continue
    chave = (dt_obj.year, dt_obj.month)
    series_mes[chave].append(int(d["nota_escala"]))

if series_mes:
    meses_ord = sorted(series_mes.keys())
    medias_mes = [np.mean(series_mes[m]) for m in meses_ord]
    rotulos_mes = [f"{a:02d}/{m:02d}" for (a, m) in meses_ord]

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(range(len(meses_ord)), medias_mes, marker="o", color="#2e75b6", linewidth=2)
    ax.axhline(np.mean(medias_mes), color="#a62019", linestyle="--", alpha=0.7,
               label=f"Média geral: {np.mean(medias_mes):.2f}")
    ax.set_xticks(range(len(meses_ord)))
    ax.set_xticklabels(rotulos_mes, rotation=90, fontsize=8)
    ax.set_ylabel("Nota média")
    ax.set_title("Evolução da Nota Média ao longo do tempo (por mês)")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Não foi possível parsear as datas para a série temporal.")

## 4. Distribuição por Ano (gráfico de pilha)

In [ ]:
anos = defaultdict(lambda: Counter())
for d in dados:
    dt_obj = parse_data(d["data"])
    if dt_obj is None:
        continue
    anos[dt_obj.year][int(d["nota_escala"])] += 1

anos_ord = sorted(anos.keys())
bottom = np.zeros(len(anos_ord))
fig, ax = plt.subplots(figsize=(11, 5))
for k in ORDENS:
    vals = [anos[a].get(k, 0) for a in anos_ord]
    ax.bar([str(a) for a in anos_ord], vals, bottom=bottom, label=ROTULOS[k], color=cores_escala[k-1])
    bottom += np.array(vals)
ax.set_ylabel("Quantidade de notas")
ax.set_xlabel("Ano")
ax.set_title("Distribuição da Nota Escala por Ano (gráfico de pilha)")
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

## 5. Distribuição por Mês (gráfico de pilha)

In [ ]:
meses_dist = defaultdict(lambda: Counter())
for d in dados:
    dt_obj = parse_data(d["data"])
    if dt_obj is None:
        continue
    chave = (dt_obj.year, dt_obj.month)
    meses_dist[chave][int(d["nota_escala"])] += 1

meses_chaves = sorted(meses_dist.keys())
rotulos_meses = [f"{a:02d}/{m:02d}" for (a, m) in meses_chaves]
bottom = np.zeros(len(meses_chaves))

fig, ax = plt.subplots(figsize=(14, 5))
for k in ORDENS:
    vals = [meses_dist[ch].get(k, 0) for ch in meses_chaves]
    ax.bar(range(len(meses_chaves)), vals, bottom=bottom, label=ROTULOS[k], color=cores_escala[k-1])
    bottom += np.array(vals)

ax.set_xticks(range(len(meses_chaves)))
ax.set_xticklabels(rotulos_meses, rotation=90, fontsize=8)
ax.set_ylabel("Quantidade de notas")
ax.set_xlabel("Mês/Ano")
ax.set_title("Distribuição da Nota Escala por Mês (gráfico de pilha)")
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

## 6. Medidas Descritivas

Estatísticas descritivas das notas da escala ordinal para caracterizar a distribuição.

In [ ]:
notas = np.array([int(d["nota_escala"]) for d in dados])

print("=" * 60)
print("   MEDIDAS DESCRITIVAS - ESCALA ORDINAL (LLM)")
print("=" * 60)
print(f"Total de notas: {len(notas)}")
print(f"Período: {min(d['data'] for d in dados)} a {max(d['data'] for d in dados)}")
print()

print("--- Estatísticas Gerais ---")
print(f"  Média aritmética:        {np.mean(notas):.3f}")
print(f"  Mediana:                 {np.median(notas):.1f}")
print(f"  Moda:                    {Counter(notas).most_common(1)[0][0]}")
print(f"  Desvio padrão:           {np.std(notas, ddof=1):.3f}")
print(f"  Coef. de variação:       {100*np.std(notas, ddof=1)/np.mean(notas):.1f}%")
print(f"  Mínimo:                  {np.min(notas)}")
print(f"  Máximo:                  {np.max(notas)}")
print(f"  Amplitude:               {np.max(notas) - np.min(notas)}")
print(f"  Q1 (25%):                {np.percentile(notas, 25):.1f}")
print(f"  Q2 (50%):                {np.percentile(notas, 50):.1f}")
print(f"  Q3 (75%):                {np.percentile(notas, 75):.1f}")
print(f"  IQR (Q3 - Q1):           {np.percentile(notas, 75) - np.percentile(notas, 25):.1f}")
print()

print("--- Distribuição de Frequências ---")
print(f"  {'Nota':<8} {'Freq. Abs.':>10} {'Freq. Rel.':>12} {'Freq. Acum.':>12}")
print("  " + "-" * 45)
freq_acum = 0
for k in ORDENS:
    c = cont.get(k, 0)
    freq_rel = 100 * c / len(notas)
    freq_acum += freq_rel
    print(f"  {k:<8} {c:>10} {freq_rel:>11.1f}% {freq_acum:>11.1f}%")
print()

print("--- Nota Média por Ano ---")
print(f"  {'Ano':<8} {'Qtd':>5} {'Média':>8} {'Desvio':>8}")
print("  " + "-" * 33)
for a in sorted(series_ano.keys()):
    arr = np.array(series_ano[a])
    print(f"  {a:<8} {len(arr):>5} {np.mean(arr):>8.2f} {np.std(arr, ddof=1) if len(arr)>1 else 0:>8.2f}")
print()

print("--- Nota Média por Mês ---")
print(f"  {'Mês':<10} {'Qtd':>5} {'Média':>8}")
print("  " + "-" * 28)
for (a, m) in sorted(series_mes.keys()):
    qtd = len(series_mes[(a, m)])
    med = np.mean(series_mes[(a, m)])
    print(f"  {a:04d}/{m:02d}  {qtd:>5} {med:>8.2f}")

## 7. Visualizações das Medidas Descritivas

Boxplot e histograma para visualizar a distribuição das notas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
bp = axes[0].boxplot(notas, vert=True, patch_artist=True, widths=0.5,
                     boxprops=dict(facecolor="#2e75b6", alpha=0.7),
                     medianprops=dict(color="#a62019", linewidth=2),
                     whiskerprops=dict(color="#333333"),
                     capprops=dict(color="#333333"),
                     flierprops=dict(markerfacecolor="#c55a11", marker="o", markersize=5))
axes[0].set_xticklabels(["Nota Escala"])
axes[0].set_ylabel("Valor")
axes[0].set_title("Boxplot da Nota Escala")
axes[0].set_yticks(ORDENS)
axes[0].set_yticklabels([ROTULOS[k] for k in ORDENS], fontsize=8)
axes[0].grid(axis="y", alpha=0.3)

stats_text = (f"Média: {np.mean(notas):.2f}\n"
              f"Mediana: {np.median(notas):.1f}\n"
              f"DP: {np.std(notas, ddof=1):.2f}\n"
              f"IQR: {np.percentile(notas, 75) - np.percentile(notas, 25):.1f}")
axes[0].text(1.15, np.mean(notas), stats_text, fontsize=9,
             verticalalignment="center", bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

# Histograma
bins = np.arange(0.5, 6, 1)
n, bins_out, patches = axes[1].hist(notas, bins=bins, color="#2e75b6", edgecolor="white",
                                    alpha=0.7, rwidth=0.85)
axes[1].set_xticks(ORDENS)
axes[1].set_xticklabels([ROTULOS[k] for k in ORDENS], rotation=15, ha="right", fontsize=8)
axes[1].set_ylabel("Frequência")
axes[1].set_title("Histograma da Nota Escala")
axes[1].grid(axis="y", alpha=0.3)

for i, (v, b) in enumerate(zip(n, patches)):
    if v > 0:
        axes[1].text(b.get_x() + b.get_width()/2, v + 0.3, str(int(v)),
                     ha="center", va="bottom", fontweight="bold", fontsize=9)

plt.tight_layout()
plt.show()

## 8. Evolução da Média Acumulada

A média acumulada ao longo do tempo revela a tendência geral do posicionamento das notas.

In [ ]:
notas_ordenadas = sorted(dados, key=lambda d: parse_data(d["data"]) or dt.datetime.min)

acumuladas = []
soma = 0
for i, d in enumerate(notas_ordenadas, 1):
    soma += int(d["nota_escala"])
    acumuladas.append(soma / i)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(len(acumuladas)), acumuladas, color="#2e75b6", linewidth=2, label="Média acumulada")
ax.axhline(np.mean(notas), color="#a62019", linestyle="--", alpha=0.7,
           label=f"Média final: {np.mean(notas):.2f}")
ax.fill_between(range(len(acumuladas)), acumuladas, alpha=0.1, color="#2e75b6")
ax.set_xlabel("Nº da nota (ordem cronológica)")
ax.set_ylabel("Nota média acumulada")
ax.set_title("Evolução da Média Acumulada ao longo do tempo")
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0.5, 5.5)
plt.tight_layout()
plt.show()